In [61]:
# functions for data conversion
import scipy.io as sio
from scipy.io.matlab import mat_struct

def _check_keys(dict):
    for key in dict:
        if isinstance(dict[key], mat_struct):
            dict[key] = _todict(dict[key])
    return dict

def _todict(matobj):
    dict = {}
    for strg in matobj._fieldnames:
        elem = matobj.__dict__[strg]
        if isinstance(elem, mat_struct):
            dict[strg] = _todict(elem)
        else:
            dict[strg] = elem
    return dict

def loadmat(filename):
    data = sio.loadmat(filename, struct_as_record=False, squeeze_me=True)
    return _check_keys(data)


In [62]:
import os
import scipy.io as sio
from scipy.io.matlab import mat_struct


def _check_keys(dict):
    for key in dict:
        if isinstance(dict[key], mat_struct):
            dict[key] = _todict(dict[key])
    return dict

def _todict(matobj):
    
    dict = {}
    for strg in matobj._fieldnames:
        elem = matobj.__dict__[strg]
        if isinstance(elem, mat_struct):
            dict[strg] = _todict(elem)
        else:
            dict[strg] = elem
    return dict

def loadmat(filename):
    data = sio.loadmat(filename, struct_as_record=False, squeeze_me=True)
    return _check_keys(data)

# --- Batch loader ---
import re  # pattern matching

def load_all_subjects(base_folder):
    all_data = {}
    
    for subj_name in os.listdir(base_folder):
        subj_path = os.path.join(base_folder, subj_name)
        # Keep only directories that start with 'Subject'
        if not os.path.isdir(subj_path) or not re.match(r'^Subject\d+$', subj_name):
            continue
        
        all_data[subj_name] = {}
        
        for cond_name in ['Control', 'Suit']:  # adjust if more conditions
            cond_path = os.path.join(subj_path, cond_name)
            mat_file = os.path.join(cond_path, 'data_katie_fixed.mat')
            
            if os.path.exists(mat_file):
                all_data[subj_name][cond_name] = loadmat(mat_file)
            else:
                print(f"Warning: {mat_file} not found")
    
    return all_data


In [63]:
# loads all data
base_folder = r"//iowa.uiowa.edu/shared/ResearchData/rdss_rvitali/Sam_Files/Research/NREIP Fire Study"
all_data = load_all_subjects(base_folder)

In [64]:
# updating keys to be [(subject, condition, activity), ...]
all_data_flat = {}

for subj in all_data.keys():
    for cond in all_data[subj].keys():
        data = all_data[subj][cond]['data']
        for activity in data.keys():
            all_data_flat[(subj, cond, activity)] = data[activity]


In [55]:
#calculates features based on sensors and subjects
import numpy as np
import pandas as pd
from scipy import signal, interpolate
import warnings
warnings.filterwarnings("ignore")

skip_subjects = {}

subject_masses = {
    "Subject1": 61.23,
    "Subject2": 78.02,
    "Subject3": 79.38,
    "Subject4": 102.06,
    "Subject5": 58.97,
    "Subject6": 63.50,
    "Subject7": 111.58,
    "Subject8": 56.7,
    "Subject9": 62.6,
    "Subject10": 77.11,
}

resting_HR = {
    "Subject1": 65,
    "Subject2": 61,
    "Subject3": 74,
    "Subject4": 70,
    "Subject5": 68,
    "Subject6": 56,
    "Subject7": 88,
    "Subject8": 80,
    "Subject9": 56,
    "Subject10": 57,
}

fs_ke = 100
fs_emg = 2000.0
fs_deriv = 2000.0
fs_hr_target = 10.0
channels = [1, 2, 11, 12]

window_sec = 0.1
samples_per_window = int(window_sec * fs_deriv)  
window_sec_hr = 1.0
samples_per_window_hr = int(window_sec_hr * fs_hr_target) 

def rms(x):
    if len(x) == 0:
        return np.nan
    return np.sign(np.mean(x)) * np.sqrt(np.mean(x**2))

def num_derivative(time, x, ver):
    if ver.lower() == 'x_1':
        version = 2
        x = np.reshape(x, (-1, 1))
    else:
        raise ValueError('Not a valid version')
        
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    delta_t = np.mean(np.diff(time))  # sampling time
    W = 4  # window size
    N = 3  # order of polynomial
    K = round(W / 2)  # center node
    if len(x) < W + 1 or np.std(x) < 1e-12:
        return np.zeros_like(x)
    
    A = np.linspace(-W, 0, W + 1) + K
    A = np.vstack([A**n for n in range(N+1)]).T
    
    R = np.zeros(N+1)
    R[1] = 1
    try:
        R = R @ np.linalg.pinv(A.T @ A) @ A.T
    except np.linalg.LinAlgError:
        # Return zero derivative if matrix is singular
        return np.zeros_like(x)
    if version != 3:
        x_d = np.zeros_like(x)
    else:
        x_d = {key: np.zeros_like(val) for key, val in x.items()}
 
    if version == 2:
        for i in range(W - K, len(x) - K):
            segment = x[i - W + K : i + K + 1, 0]
            if np.any(np.isnan(segment)):
                x_d[i, :] = 0.0
            else:
                x_d[i, :] = (R @ segment) / delta_t

    return x_d

b_emg_deriv, a_emg_deriv = signal.butter(4, [50/(fs_deriv/2), 150/(fs_deriv/2)], btype='band')
b_hr, a_hr = signal.butter(2, [0.01/(fs_hr_target/2), 0.5/(fs_hr_target/2)], btype='band')

all_rows = []
print("Processing all subjects and activities...")

for subj, cond, act in all_data_flat.keys():
    if subj in skip_subjects:
        continue
    
    print(f"{subj} | {cond} | {act}")

    data = all_data_flat[(subj, cond, act)]

    emg_time = data['EMG'][:, 0]
    t_start, t_end = emg_time[0], emg_time[-1]
    
    bin_edges = np.arange(t_start, t_end, 15.0)
    if t_end - bin_edges[-1] > 10:
        bin_edges = np.append(bin_edges, bin_edges[-1] + 15.0)
    else:
        bin_edges = np.append(bin_edges, t_end)

    subject_mass = subject_masses[subj]
    rest_hr = resting_HR[subj] 
    mNorm = subject_mass / 82.2

    segments = [
        {'Name':'W','I':0.02654,'mass':0,'dist':0,'cols':[3,4,5],'src':'back'},
        {'Name':'C','I':0.33201,'mass':0,'dist':0,'cols':[3,4,5],'src':'neck'},
        {'Name':'AL','I':0.06397,'mass':0,'dist':0,'cols':[3,4,5],'src':'lshank'},
        {'Name':'AR','I':0.06271,'mass':0,'dist':0,'cols':[3,4,5],'src':'rshank'}
    ]

    KE_df = pd.DataFrame()
    for s in segments:
        acc = data['IMU'][s['src']]
        x,y,z = [np.nan_to_num(acc[:,c]) for c in s['cols']]
        omega = np.sqrt(x**2 + y**2 + z**2)
        KE_df[s['Name']] = 0.5 * (s['I']*mNorm) * omega**2

    KE_df['total'] = KE_df.sum(axis=1)

    metab = data['HRandTemp']
    hr_time = metab[:,0]
    hr = metab[:,4]

    t_hr = np.linspace(hr_time[0], hr_time[-1], int((hr_time[-1] - hr_time[0]) * fs_hr_target))
    hr_interp = interpolate.interp1d(hr_time, hr, fill_value="extrapolate")(t_hr)
    hr_filt = signal.filtfilt(b_hr, a_hr, hr_interp)

    emg = data['EMG']
    emg_der = {}
    emg_raw = {}

    for ch in channels:
        raw = np.nan_to_num(emg[:,ch])
        emg_raw[ch] = raw  # Store raw EMG
        emg_der[ch] = np.nan_to_num(signal.filtfilt(b_emg_deriv, a_emg_deriv, raw))

    for i in range(len(bin_edges)-1):
        t0, t1 = bin_edges[i], bin_edges[i+1]

        imu_time = data['IMU']['Timestamps']
        ke0 = np.searchsorted(imu_time, t0)
        ke1 = np.searchsorted(imu_time, t1)

        emg0, emg1 = np.searchsorted(emg_time,t0), np.searchsorted(emg_time,t1)
        
        hr0 = np.searchsorted(hr_time, t0)
        hr1 = np.searchsorted(hr_time, t1)

        row = {
            'Subject':subj, 'Condition':cond, 'Activity':act,
            'Time':t0, 'Weight':subject_mass,
            'KE':np.nanmean(KE_df['total'][ke0:ke1])
        }

        row['HR'] = np.nanmean(hr[(hr_time>=t0)&(hr_time<t1)] - rest_hr)

        # HR Derivative
        start_idx_hr = hr0
        end_idx_hr = hr1
        deriv_values_hr = []
        
        for window_start in range(start_idx_hr, end_idx_hr, samples_per_window_hr):
            window_end = min(window_start + samples_per_window_hr, end_idx_hr)
            sig_bin_hr = hr_filt[window_start:window_end]
            t_bin_local_hr = np.arange(len(sig_bin_hr)) / fs_hr_target
            deriv = num_derivative(t_bin_local_hr, sig_bin_hr, 'x_1')
            deriv = deriv.flatten()
            deriv_values_hr.append(deriv.mean())
        
        row['HRDeriv'] = rms(np.array(deriv_values_hr))

        start_idx_emg = emg0
        end_idx_emg = emg1
        
        # Calculate mean raw EMG for each channel in this bin (convert mV to V)
        for ch in channels:
            row[f'EMG{ch}Raw'] = np.nanmean(emg_raw[ch][emg0:emg1]) / 1000.0
        
        # Calculate derivative features for each channel
        for ch in channels:
            deriv_values_in_bin = []
            for window_start in range(start_idx_emg, end_idx_emg, samples_per_window):
                window_end = min(window_start + samples_per_window, end_idx_emg)
                sig_bin = emg_der[ch][window_start:window_end]
                t_bin_local = np.arange(len(sig_bin)) / fs_deriv
                sig_bin = np.nan_to_num(sig_bin, nan=0.0, posinf=0.0, neginf=0.0)
                deriv = num_derivative(t_bin_local, sig_bin, 'x_1')
                deriv = deriv.flatten()
                deriv_values_in_bin.append(deriv.mean())
            
            row[f'EMG{ch}Deriv'] = np.array(deriv_values_in_bin).mean()

        all_rows.append(row)

merged_df = pd.DataFrame(all_rows)

merged_df = merged_df.rename(columns={
    'EMG1Deriv':'rsolDeriv','EMG2Deriv':'lsolDeriv',
    'EMG11Deriv':'rbfDeriv','EMG12Deriv':'lbfDeriv',
    'EMG1Raw':'rsol','EMG2Raw':'lsol',
    'EMG11Raw':'rbf','EMG12Raw':'lbf'
})

final_cols = ['Subject','Condition','Activity','Time','KE','HR','HRDeriv','Weight',
              'rsol','lsol','rbf','lbf',
              'rsolDeriv','lsolDeriv','rbfDeriv','lbfDeriv']

merged_df = merged_df[final_cols]

print("Done:", merged_df.shape)

Processing all subjects and activities...
Subject1 | Control | Jog
Subject1 | Control | Wheel
Subject1 | Control | Fire
Subject1 | Control | Dummy
Subject1 | Control | Stairs
Subject1 | Control | Walk
Subject1 | Suit | Jog
Subject1 | Suit | Wheel
Subject1 | Suit | Fire
Subject1 | Suit | Dummy
Subject1 | Suit | Stairs
Subject1 | Suit | Walk
Subject2 | Control | Jog
Subject2 | Control | Wheel
Subject2 | Control | Fire
Subject2 | Control | Dummy
Subject2 | Control | Stairs
Subject2 | Control | Walk
Subject2 | Suit | Jog
Subject2 | Suit | Wheel
Subject2 | Suit | Fire
Subject2 | Suit | Dummy
Subject2 | Suit | Stairs
Subject2 | Suit | Walk
Subject3 | Control | Jog
Subject3 | Control | Wheel
Subject3 | Control | Fire
Subject3 | Control | Dummy
Subject3 | Control | Stairs
Subject3 | Control | Walk
Subject3 | Suit | Jog
Subject3 | Suit | Wheel
Subject3 | Suit | Fire
Subject3 | Suit | Dummy
Subject3 | Suit | Stairs
Subject3 | Suit | Walk
Subject4 | Control | Jog
Subject4 | Control | Wheel
Subjec

In [65]:
# subject 10 only, used because subject 10 came much later in the studies; 
# it would be easier to incorporate this subject into the previous block
import numpy as np
import pandas as pd
from scipy import signal, interpolate
import warnings
warnings.filterwarnings("ignore")


subject_masses = {
    "Subject1": 61.23, "Subject2": 78.02, "Subject3": 79.38, "Subject4": 102.06,
    "Subject5": 58.97, "Subject6": 63.50, "Subject7": 111.58, "Subject8": 56.7,
    "Subject9": 62.6, "Subject10": 77.11,
}

resting_HR = {
    "Subject1": 65, "Subject2": 61, "Subject3": 74, "Subject4": 70,
    "Subject5": 68, "Subject6": 56, "Subject7": 88, "Subject8": 80,
    "Subject9": 56, "Subject10": 57,
}

fs_ke = 100
fs_emg = 2000.0
fs_deriv = 2000.0
fs_hr_target = 10.0
channels = [1, 2, 11, 12]

window_sec = 0.1
samples_per_window = int(window_sec * fs_deriv)  
window_sec_hr = 1.0
samples_per_window_hr = int(window_sec_hr * fs_hr_target) 

# --- Helper Functions ---
def rms(x):
    if len(x) == 0:
        return np.nan
    return np.sign(np.mean(x)) * np.sqrt(np.mean(x**2))

def num_derivative(time, x, ver):
    if ver.lower() == 'x_1':
        version = 2
        x = np.reshape(x, (-1, 1))
    else:
        raise ValueError('Not a valid version')
        
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    delta_t = np.mean(np.diff(time))
    W, N = 4, 3
    K = round(W / 2)
    if len(x) < W + 1 or np.std(x) < 1e-12:
        return np.zeros_like(x)
    
    A = np.linspace(-W, 0, W + 1) + K
    A = np.vstack([A**n for n in range(N+1)]).T
    
    R = np.zeros(N+1)
    R[1] = 1
    try:
        R = R @ np.linalg.pinv(A.T @ A) @ A.T
    except np.linalg.LinAlgError:
        return np.zeros_like(x)

    x_d = np.zeros_like(x)
    for i in range(W - K, len(x) - K):
        segment = x[i - W + K : i + K + 1, 0]
        if not np.any(np.isnan(segment)):
            x_d[i, :] = (R @ segment) / delta_t
    return x_d

# --- Filters ---
b_emg_deriv, a_emg_deriv = signal.butter(4, [50/(fs_deriv/2), 150/(fs_deriv/2)], btype='band')
b_hr, a_hr = signal.butter(2, [0.01/(fs_hr_target/2), 0.5/(fs_hr_target/2)], btype='band')

# --- Processing Loop (Subject 10 Only) ---
all_rows_subj10 = []
print("Processing Subject 10 data...")

for (subj, cond, act) in all_data_flat.keys():
    # Only process Subject 10
    if subj != "Subject10":
        continue
    
    print(f"Current segment: {subj} | {cond} | {act}")
    data = all_data_flat[(subj, cond, act)]

    # Time and Bins
    emg_time = data['EMG'][:, 0]
    t_start, t_end = emg_time[0], emg_time[-1]
    bin_edges = np.append(np.arange(t_start, t_end, 15.0), t_end)

    subject_mass = subject_masses[subj]
    rest_hr = resting_HR[subj] 
    mNorm = subject_mass / 82.2

    # Kinetic Energy Calculation
    segments_imu = [
        {'Name':'W','I':0.02654,'src':'back'},
        {'Name':'C','I':0.33201,'src':'neck'},
        {'Name':'AL','I':0.06397,'src':'lshank'},
        {'Name':'AR','I':0.06271,'src':'rshank'}
    ]

    KE_df = pd.DataFrame()
    for s in segments_imu:
        acc = data['IMU'][s['src']]
        # Accessing cols 3,4,5 (wx, wy, wz) for angular velocity omega
        omega = np.sqrt(np.sum(np.nan_to_num(acc[:, 3:6])**2, axis=1))
        KE_df[s['Name']] = 0.5 * (s['I']*mNorm) * omega**2
    KE_df['total'] = KE_df.sum(axis=1)

    # HR Processing
    metab = data['HRandTemp']
    hr_time, hr = metab[:,0], metab[:,4]
    t_hr = np.linspace(hr_time[0], hr_time[-1], int((hr_time[-1] - hr_time[0]) * fs_hr_target))
    hr_interp = interpolate.interp1d(hr_time, hr, fill_value="extrapolate")(t_hr)
    hr_filt = signal.filtfilt(b_hr, a_hr, hr_interp)

    # EMG Processing
    emg = data['EMG']
    emg_der, emg_raw = {}, {}
    for ch in channels:
        raw = np.nan_to_num(emg[:,ch])
        emg_raw[ch] = raw
        emg_der[ch] = np.nan_to_num(signal.filtfilt(b_emg_deriv, a_emg_deriv, raw))

    # Binning and Feature Extraction
    for i in range(len(bin_edges)-1):
        t0, t1 = bin_edges[i], bin_edges[i+1]
        
        imu_time = data['IMU']['Timestamps']
        ke0, ke1 = np.searchsorted(imu_time, t0), np.searchsorted(imu_time, t1)
        emg0, emg1 = np.searchsorted(emg_time, t0), np.searchsorted(emg_time, t1)
        hr0, hr1 = np.searchsorted(hr_time, t0), np.searchsorted(hr_time, t1)

        row = {
            'Subject':subj, 'Condition':cond, 'Activity':act,
            'Time':t0, 'Weight':subject_mass,
            'KE':np.nanmean(KE_df['total'][ke0:ke1]),
            'HR':np.nanmean(hr[(hr_time>=t0)&(hr_time<t1)] - rest_hr)
        }

        # HR Derivative (RMS of windowed derivatives)
        deriv_values_hr = []
        for w_start in range(hr0, hr1, samples_per_window_hr):
            w_end = min(w_start + samples_per_window_hr, hr1)
            sig_bin_hr = hr_filt[w_start:w_end]
            if len(sig_bin_hr) > 5:
                t_bin_hr = np.arange(len(sig_bin_hr)) / fs_hr_target
                deriv_values_hr.append(num_derivative(t_bin_hr, sig_bin_hr, 'x_1').mean())
        row['HRDeriv'] = rms(np.array(deriv_values_hr))

        # EMG Features
        for ch in channels:
            row[f'EMG{ch}Raw'] = np.nanmean(emg_raw[ch][emg0:emg1]) / 1000.0
            deriv_values_emg = []
            for w_start in range(emg0, emg1, samples_per_window):
                w_end = min(w_start + samples_per_window, emg1)
                sig_bin = emg_der[ch][w_start:w_end]
                if len(sig_bin) > 5:
                    t_bin = np.arange(len(sig_bin)) / fs_deriv
                    deriv_values_emg.append(num_derivative(t_bin, sig_bin, 'x_1').mean())
            row[f'EMG{ch}Deriv'] = np.array(deriv_values_emg).mean()

        all_rows_subj10.append(row)

# --- Final Assembly and Append ---
new_data_df = pd.DataFrame(all_rows_subj10)

# Renaming to match existing schema
new_data_df = new_data_df.rename(columns={
    'EMG1Deriv':'rsolDeriv','EMG2Deriv':'lsolDeriv',
    'EMG11Deriv':'rbfDeriv','EMG12Deriv':'lbfDeriv',
    'EMG1Raw':'rsol','EMG2Raw':'lsol',
    'EMG11Raw':'rbf','EMG12Raw':'lbf'
})

final_cols = ['Subject','Condition','Activity','Time','KE','HR','HRDeriv','Weight',
              'rsol','lsol','rbf','lbf',
              'rsolDeriv','lsolDeriv','rbfDeriv','lbfDeriv']

# Reorder columns and filter
new_data_df = new_data_df[final_cols]

# Append Subject 10 to the existing merged_df
merged_df = pd.concat([merged_df, new_data_df], ignore_index=True)

print(f"Subject 10 processing complete. Final dataframe shape: {merged_df.shape}")

Processing Subject 10 data...
Current segment: Subject10 | Control | Jog
Current segment: Subject10 | Control | Wheel
Current segment: Subject10 | Control | Fire
Current segment: Subject10 | Control | Dummy
Current segment: Subject10 | Control | Stairs
Current segment: Subject10 | Control | Walk
Current segment: Subject10 | Suit | Jog
Current segment: Subject10 | Suit | Wheel
Current segment: Subject10 | Suit | Fire
Current segment: Subject10 | Suit | Dummy
Current segment: Subject10 | Suit | Stairs
Current segment: Subject10 | Suit | Walk
Subject 10 processing complete. Final dataframe shape: (1159, 16)


In [68]:
# converts from deg to rad
merged_df['KE'] = merged_df['KE'] * (np.pi/180)**2

print(merged_df.head(1500))

        Subject Condition Activity   Time            KE         HR   HRDeriv  \
0      Subject1   Control      Jog  152.5  6.653712e-11  43.533333 -0.229687   
1      Subject1   Control      Jog  167.5  6.756200e-11  53.533333  0.659470   
2      Subject1   Control      Jog  182.5  6.621147e-11  67.733333  1.055165   
3      Subject1   Control      Jog  197.5  7.122166e-11  73.866667  0.826932   
4      Subject1   Control      Jog  212.5  7.104483e-11  76.400000  0.761316   
...         ...       ...      ...    ...           ...        ...       ...   
1154  Subject10      Suit     Walk  904.5  1.670583e-11  60.732947  0.304285   
1155  Subject10      Suit     Walk  919.5  1.679985e-11  58.659529  0.032233   
1156  Subject10      Suit     Walk  934.5  1.684121e-11  58.943267 -0.068843   
1157  Subject10      Suit     Walk  949.5  1.672505e-11  60.600024 -0.247099   
1158  Subject10      Suit     Walk  964.5  1.687665e-11  59.861869  0.050589   

      Weight      rsol      lsol       

In [69]:
# interpolates subject6 HR and HRDeriv data for 3 rows, where sensors cut out in stairs
merged_df[['HR', 'HRDeriv']] = merged_df[['HR', 'HRDeriv']].interpolate(method='linear')

In [70]:
# Save DataFrame to pickle file
merged_df.to_pickle("/Users/katbutler/MetabolicCost/merged_test_df.pkl")